# Fine-tuning Kompress v2 on Ultrawhale Q&A pairs

**Goal**: Fine-tune [chopratejas/kompress-v2-base](https://huggingface.co/chopratejas/kompress-v2-base) to better preserve semantically essential tokens in agent loop traffic.

**The Sapir-Whorf connection**: Language shapes thought. Bad compression removes tokens the agent needs to reason. This fine-tune calibrates which tokens are linguistically irreplaceable.

**Runtime**: ~45 min on Colab T4 (free tier) or ~15 min on RTX 4090.

**Cost on vast.ai**: $0.18 (RTX 4090 @ $0.356/hr, 30 min)

---

**What you'll build**:
1. Silver-labeled training data from ultrawhale Q&A pairs
2. LoRA fine-tune of ModernBERT heads
3. ONNX export for headroom proxy
4. Live compression demo

In [ ]:
# Install dependencies
!pip install -q transformers peft accelerate datasets huggingface_hub requests torch

## 1. Load and label the ultrawhale dataset

The ultrawhale dataset has two responses per question:
- `deepseek_response`: verbose (~300 tokens)
- `free_response`: concise (~80 tokens)

Silver labels: token in verbose gets label=1 if the word appears in the concise version.

In [ ]:
import json, re, requests
from pathlib import Path

HF_BASE = "https://huggingface.co/datasets/PeetPedro/ultrawhale-dogfood/resolve/main"
JSONL_FILES = ["dogfeed-v1-initial.jsonl", "dogfeed-v2-science.jsonl", "dogfeed-v3-enriched.jsonl"]

_MUST_KEEP = re.compile(
    r"\d+(\.\d+)?|[A-Z_]{2,}|[a-z_]+\.[a-z_]+|/[a-z/._-]{2,}|\.[a-z]{2,4}\b|--?[a-z][\w-]*|\b[A-Z][a-z]+[A-Z]\w*"
)

def load_pairs():
    pairs = []
    for fname in JSONL_FILES:
        r = requests.get(f"{HF_BASE}/{fname}", timeout=30)
        for line in r.text.splitlines():
            if not line.strip(): continue
            d = json.loads(line)
            verbose = (d.get("deepseek_response") or "").strip()
            compressed = (d.get("free_response") or "").strip()
            if not verbose or not compressed: continue
            ratio = len(verbose) / max(len(compressed), 1)
            if ratio < 1.3 or len(verbose) < 80: continue
            pairs.append({"text": verbose, "reference": compressed})
    return pairs

pairs = load_pairs()
print(f"Loaded {len(pairs)} training pairs")
print("\nExample:")
print("VERBOSE:", pairs[0]['text'][:200])
print("\nCOMPRESSED:", pairs[0]['reference'][:150])

## 2. The model architecture

Kompress = ModernBERT-base + two custom heads. We load v2 weights and apply LoRA to the last 4 encoder layers.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from peft import get_peft_model, LoraConfig

BASE_ENCODER = "answerdotai/ModernBERT-base"

class HeadroomCompressorModel(nn.Module):
    """Dual-head ModernBERT: token classifier + span CNN."""
    
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(BASE_ENCODER, attn_implementation="eager")
        H = self.encoder.config.hidden_size  # 768
        self.token_dropout = nn.Dropout(0.1)
        self.token_head = nn.Linear(H, 2)  # Head 1: keep/discard
        self.span_conv = nn.Sequential(  # Head 2: span importance
            nn.Conv1d(H, 256, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(256, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )
    
    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids, attention_mask=attention_mask).last_hidden_state
        logits = self.token_head(self.token_dropout(h))        # [B, L, 2]
        span   = self.span_conv(h.transpose(1,2)).squeeze(1)  # [B, L]
        return logits, span
    
    def compress(self, text, tokenizer, threshold=0.5):
        """Compress text, return (compressed_text, keep_rate)."""
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            logits, span = self(enc["input_ids"], enc["attention_mask"])
            probs = torch.softmax(logits, dim=-1)[0, :, 1]
            span_s = span[0]
            borderline = (probs > 0.3) & (probs <= threshold)
            keep = (probs > threshold) | (borderline & (span_s > 0.5))
        tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
        kept = [t for t, k in zip(tokens, keep.tolist()) 
                if k and t not in ["[CLS]", "[SEP]", "<s>", "</s>"]]
        return tokenizer.convert_tokens_to_string(kept), keep.float().mean().item()

def load_v2_weights(model):
    from huggingface_hub import hf_hub_download
    path = hf_hub_download("chopratejas/kompress-v2-base", "merged.pt")
    ckpt = torch.load(path, map_location="cpu", weights_only=True)
    model.encoder.load_state_dict(ckpt["encoder_state_dict"], strict=False)
    model.token_head.load_state_dict(ckpt["token_head_state_dict"])
    model.span_conv.load_state_dict(ckpt["span_head_state_dict"])
    print("v2 weights loaded")

tokenizer = AutoTokenizer.from_pretrained(BASE_ENCODER)
model = HeadroomCompressorModel()
load_v2_weights(model)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Demo: compress a sample BEFORE fine-tuning (baseline)
sample_text = pairs[0]['text']
compressed, keep_rate = model.compress(sample_text, tokenizer)
print(f"ORIGINAL ({len(sample_text.split())} words):")
print(sample_text[:300])
print(f"\nCOMPRESSED ({keep_rate:.1%} tokens kept):")
print(compressed[:300])
print(f"\nREFERENCE:")
print(pairs[0]['reference'][:300])

## 3. Apply LoRA to last 4 layers

Only 2M of 140M params become trainable. Encoder backbone stays frozen — it already knows language.

In [ ]:
n_layers = model.encoder.config.num_hidden_layers  # 22
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "key", "value"],
    lora_dropout=0.05,
    bias="none",
    layers_to_transform=list(range(n_layers - 4, n_layers)),
)
model.encoder = get_peft_model(model.encoder, lora_cfg)
model.encoder.print_trainable_parameters()

## 4. Build the dataset with silver labels

**Silver label rule** (Sapir-Whorf criterion):
- Token gets label=1 (keep) if it appears in the compressed reference
- Override to label=1 with weight=3.0: numbers, identifiers, paths, flags
- These are the tokens the agent **cannot lose** without losing the concept

In [ ]:
from torch.utils.data import Dataset, DataLoader

def silver_labels(token_strings, ref_words):
    labels, weights = [], []
    for tok in token_strings:
        clean = re.sub(r"[^\w]", "", tok).lower()
        must = bool(_MUST_KEEP.search(tok))
        in_ref = clean in ref_words or len(clean) < 3
        label = 1 if (must or in_ref) else 0
        labels.append(label)
        weights.append(3.0 if must else (1.0 if label == 1 else 0.5))
    return labels, weights

class KompressDS(Dataset):
    def __init__(self, pairs, tokenizer, max_len=512):
        self.data = pairs
        self.tok = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        d = self.data[i]
        ref_words = set(re.findall(r"\b[a-z]{3,}\b", d['reference'].lower()))
        enc = self.tok(d['text'], max_length=self.max_len, truncation=True,
                       padding=False, return_offsets_mapping=True, return_tensors="pt")
        ids = enc['input_ids'][0]
        mask = enc['attention_mask'][0]
        offsets = enc['offset_mapping'][0].tolist()
        toks = [d['text'][s:e] if e>s else self.tok.decode([ids[j]]) for j,(s,e) in enumerate(offsets)]
        labs, wts = silver_labels(toks, ref_words)
        for j in range(len(labs)):
            if offsets[j] == (0,0): labs[j]=1; wts[j]=0.0  # special tokens
        return {'input_ids': ids, 'attention_mask': mask,
                'labels': torch.tensor(labs, dtype=torch.long),
                'weights': torch.tensor(wts, dtype=torch.float)}

def collate(batch):
    L = max(b['input_ids'].shape[0] for b in batch)
    B = len(batch)
    ids = torch.zeros(B,L,dtype=torch.long)
    mask = torch.zeros(B,L,dtype=torch.long)
    labs = torch.zeros(B,L,dtype=torch.long)
    wts  = torch.zeros(B,L,dtype=torch.float)
    for i,b in enumerate(batch):
        n = b['input_ids'].shape[0]
        ids[i,:n]=b['input_ids']; mask[i,:n]=b['attention_mask']
        labs[i,:n]=b['labels'];   wts[i,:n]=b['weights']
    return {'input_ids':ids,'attention_mask':mask,'labels':labs,'weights':wts}

import random
random.shuffle(pairs)
split = int(len(pairs)*0.9)
train_ds = KompressDS(pairs[:split], tokenizer)
test_ds  = KompressDS(pairs[split:], tokenizer)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate)
print(f"Train: {len(train_ds)}, Test: {len(test_ds)}, Steps/epoch: {len(train_dl)}")

## 5. Train

In [ ]:
EPOCHS = 3
DEVICE = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Training on: {DEVICE}")
model.to(DEVICE)

trainable = (
    list(model.encoder.parameters())
    + list(model.token_head.parameters())
    + list(model.span_conv.parameters())
)
opt = torch.optim.AdamW(trainable, lr=2e-4, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS*len(train_dl))
criterion = nn.CrossEntropyLoss(reduction='none')

for ep in range(EPOCHS):
    model.train()
    total = 0.0
    for step, batch in enumerate(train_dl):
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        labs  = batch['labels'].to(DEVICE)
        wts   = batch['weights'].to(DEVICE)
        
        logits, span = model(ids, mask)
        B,L,_ = logits.shape
        
        tok_loss = criterion(logits.view(B*L,2), labs.view(B*L))
        tok_loss = (tok_loss * wts.view(B*L) * mask.view(B*L).float()).sum()
        tok_loss = tok_loss / (mask.float().sum() + 1e-8)
        
        with torch.no_grad():
            target = (torch.softmax(logits,dim=-1)[:,:,1] > 0.5).float()
        span_loss = nn.functional.binary_cross_entropy(
            span*mask.float(), target*mask.float(), reduction='mean')
        
        loss = tok_loss + 0.3*span_loss
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(trainable, 1.0)
        opt.step(); sched.step()
        total += loss.item()
        
        if step % 20 == 0:
            print(f"Ep {ep+1}/{EPOCHS} step {step}/{len(train_dl)} loss={loss.item():.4f}")
    
    print(f"=== Epoch {ep+1} avg loss: {total/len(train_dl):.4f} ===")

## 6. Evaluate

In [ ]:
model.eval()
keep_rates, exact_pcts = [], []

for item in test_ds.data[:100]:  # sample 100
    compressed, kr = model.compress(item['text'], tokenizer)
    # Exact preservation: % of must-keep tokens that survived
    must = set(m[0] for m in _MUST_KEEP.findall(item['text']))
    if must:
        ep = sum(1 for t in must if t in compressed) / len(must)
    else:
        ep = 1.0
    keep_rates.append(kr)
    exact_pcts.append(ep)

print(f"Avg keep_rate: {sum(keep_rates)/len(keep_rates):.4f}  (v2 baseline: 0.8097)")
print(f"Avg exact_pct: {sum(exact_pcts)/len(exact_pcts):.4f}  (must-keep survival)")

## 7. Demo: before vs after

In [ ]:
# Load v2 baseline for comparison
baseline = HeadroomCompressorModel()
load_v2_weights(baseline)
baseline.eval()

# Use a technical text with identifiers that matter
test_text = """
The SIGILL error occurs when the CPU encounters an illegal instruction.
On NixOS with Caddy 2.11.x (Go 1.22+), the SystemCallFilter=@system-service 
blocks the rseq syscall that the Go runtime uses, causing a SIGSYS crash.
The fix is to add SystemCallFilter = lib.mkForce [] in the hardening.nix file
to disable seccomp filtering entirely. This resolves the crash loop.
""".strip()

before, kr_before = baseline.compress(test_text, tokenizer)
after,  kr_after  = model.compress(test_text, tokenizer)

print(f"ORIGINAL ({len(test_text.split())} words):")
print(test_text)
print(f"\nBEFORE fine-tune (keep={kr_before:.1%}):")
print(before)
print(f"\nAFTER fine-tune (keep={kr_after:.1%}):")
print(after)

## 8. Save and export ONNX

In [ ]:
from pathlib import Path

# Merge LoRA
model.encoder = model.encoder.merge_and_unload()

# Save checkpoint
Path("kompress-v3").mkdir(exist_ok=True)
tokenizer.save_pretrained("kompress-v3")
torch.save({
    "encoder_state_dict": model.encoder.state_dict(),
    "token_head_state_dict": model.token_head.state_dict(),
    "span_head_state_dict": model.span_conv.state_dict(),
}, "kompress-v3/merged.pt")
print("Saved to kompress-v3/")

# Export ONNX
!pip install -q onnx onnxruntime

class Wrapper(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        logits, span = self.m(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=-1)[:,:,1]
        return probs * (0.5 + 0.5 * span)

Path("kompress-v3/onnx").mkdir(exist_ok=True)
dummy = tokenizer("hello world", return_tensors="pt")
torch.onnx.export(
    Wrapper(model), (dummy["input_ids"], dummy["attention_mask"]),
    "kompress-v3/onnx/kompress-fp32.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["final_scores"],
    dynamic_axes={"input_ids":{0:"b",1:"s"}, "attention_mask":{0:"b",1:"s"}, "final_scores":{0:"b",1:"s"}},
    opset_version=17,
)
print("ONNX exported")

In [ ]:
# Optional: upload to HuggingFace
# Set your token: HF_TOKEN = "hf_xxx"

HF_TOKEN = ""  # fill in or set as env var
HF_REPO  = "your-username/kompress-v3"

if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO, exist_ok=True)
    api.upload_folder(
        folder_path="kompress-v3",
        repo_id=HF_REPO,
        commit_message="kompress-v3: LoRA fine-tuned on ultrawhale + headroom-aware labels",
    )
    print(f"Uploaded to huggingface.co/{HF_REPO}")
else:
    print("Set HF_TOKEN to upload. Model saved locally in kompress-v3/")

## Additional datasets to explore

| Dataset | Why useful | Pair structure |
|---|---|---|
| `teknium/OpenHermes-2.5` | 1M GPT-4 instruction pairs | long/short response contrast |
| `Open-Orca/OpenOrca` | GPT-3.5 vs GPT-4 same question | verbosity contrast |
| `FrancophonIA/WikiSum` | Wikipedia article vs abstract | extreme length ratio |
| `theblackcat102/sharegpt` | Real ChatGPT conversations | multi-turn clarification |
| `code_search_net` | Code + docstring | code compression domain |
| `billsum` | Legal bills vs summaries | high-density technical language |

Load any of them with:
```python
from datasets import load_dataset
ds = load_dataset("Open-Orca/OpenOrca", split="train", streaming=True)
```

Use the same silver label approach: shorter version = reference, longer version = training text.

## Running on vast.ai (RTX 4090, $0.356/hr)

```bash
# Find instance
vastai search offers 'gpu_name=RTX_4090 num_gpus=1' --order dph_total --limit 3

# Rent and run (~30 min, ~$0.18)
vastai create instance <OFFER_ID> \
    --image pytorch/pytorch:2.3.0-cuda12.1-cudnn8-runtime \
    --disk 30 \
    --env "-e HF_TOKEN=your_token" \
    --onstart "bash /workspace/ultrawhale/scripts/vast_setup.sh && \
               bash /workspace/ultrawhale/scripts/run_training.sh"
```

Or with task:
```bash
task kompress-vast-launch OFFER_ID=37031007
```

---
## Experiment A: Self-labeled references (v4)

**Hypothesis:** If we use kompress-v3 + the hard override to compress the training texts,
the compressed output preserves all must-keep tokens (mk_in_ref → 1.0).
Training on these self-labeled pairs should push exact_keep_pct above the 0.882 ceiling.

**Result to check:** does exact_pct on heretic-style prompts improve beyond 0.942?


In [ ]:
# Self-label training data using v3 + override
# This runs on GPU — use vast.ai or local CUDA

import json, re, torch
from train_kompress import HeadroomCompressorModel, load_v2_weights
from transformers import AutoTokenizer

_MUST_KEEP_RE = re.compile(
    r'\b0x[0-9A-Fa-f]+\b'
    r'|(?<![\w.])\d+(?:\.\d+)?(?![\w.])'
    r'|[A-Z_]{2,}'
    r'|[a-z_][a-z0-9_]*\.[a-z0-9_]+'
    r'|/[a-z0-9/._-]{2,}'
    r'|\.[a-z]{2,4}\b'
    r'|--?[a-z][\w-]*'
    r'|\b[A-Z][a-z]+[A-Z]\w*'
)

BASE = 'answerdotai/ModernBERT-base'
tok = AutoTokenizer.from_pretrained(BASE)
model = HeadroomCompressorModel(BASE)
load_v2_weights(model, 'PeetPedro/kompress-v3')
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'Device: {device}')

def compress_with_override(text):
    enc = tok(text, return_tensors='pt', truncation=True, max_length=512)
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits, span = model(enc['input_ids'], enc['attention_mask'])
        probs = torch.softmax(logits, dim=-1)[0, :, 1]
        scores = probs * (0.5 + 0.5 * span[0])
        keep = scores > 0.5
    tokens = tok.convert_ids_to_tokens(enc['input_ids'][0])
    for i, t in enumerate(tokens):
        w = tok.convert_tokens_to_string([t]).strip()
        if _MUST_KEEP_RE.search(w): keep[i] = True
    kept = [t for t, k in zip(tokens, keep)
            if k and t not in ('[CLS]','[SEP]','<s>','</s>')]
    return tok.convert_tokens_to_string(kept)

# Self-label the training split
records = [json.loads(l) for l in open('data/kompress_train_split.jsonl')]
out = []
for i, r in enumerate(records):
    if i % 200 == 0: print(f'{i}/{len(records)}')
    new_ref = compress_with_override(r['text'])
    ratio = len(r['text']) / max(len(new_ref), 1)
    if ratio >= 1.2 and len(new_ref) >= 30:
        out.append({**r, 'reference': new_ref, 'source': 'self_labeled_v3_override'})

# Check mk_in_ref
import random; random.seed(42)
samp = random.sample(out, min(100, len(out)))
mk_t = mk_r = 0
for r in samp:
    must = [m.group(0) for m in _MUST_KEEP_RE.finditer(r['text'])]
    mk_t += len(must)
    mk_r += sum(1 for m in must if m in r['reference'])
print(f'mk_in_ref: {mk_r/max(mk_t,1):.3f} (ultrawhale baseline: ~0.72)')
print(f'Pairs: {len(out)}')

with open('data/self_labeled_train.jsonl', 'w') as f:
    for r in out: f.write(json.dumps(r, ensure_ascii=False) + '\n')


## Experiment C: Full benchmark — all versions

Compares v2-base through v4 on two datasets:
- **Q&A test set** (ultrawhale 10% holdout, noisy labels)
- **Heretic technical** (adversarial: dense must-keep tokens)

Expected finding: Q&A exact_pct shows a flat ceiling (~0.882); heretic
exact_pct reveals the model's real capability and the override's impact.


In [ ]:
# Full benchmark across all versions
# Run: python eval_kompress.py --model PeetPedro/kompress-v4 --data data/kompress_test.jsonl

VERSIONS = [
    ('v2-base', 'chopratejas/kompress-v2-base'),
    ('v3',      'PeetPedro/kompress-v3'),
    ('v3.1',    'PeetPedro/kompress-v31'),
    ('v3.2',    'PeetPedro/kompress-v32'),
    ('v3.3',    'PeetPedro/kompress-v33'),
    ('v4',      'PeetPedro/kompress-v4'),
]

# Results table (fill after running)
# Version | Q&A keep_rate | Q&A exact_pct | Heretic exact_pct | Heretic+override
# v2-base |       0.810   |      —        |        —          |       —
# v3      |       0.728   |    0.882      |      0.942        |     0.969
# v3.1    |       0.718   |    0.878      |        ?          |       ?
# v3.2    |       0.713   |    0.877      |        ?          |       ?
# v3.3    |       0.724   |    0.879      |        ?          |       ?
# v4      |         ?     |      ?        |        ?          |       ?
print('Run eval_kompress.py for each version and fill the table above')
